# Alpha-Width Grid SGD Experiments

Train parity networks over a grid of readout-scale `alpha` and width `N`, then analyze train/test curves, heatmaps, PCA rank reduction, and trained weight variances.

## Setup

Mount Google Drive, clone the public repo, install it in editable mode, and define run/plot/analysis directories.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/labofdoubt/feature-learning-parity-task.git"
REPO_DIR = Path("/content/feature-learning-parity-task")

DRIVE_ROOT = Path("/content/drive/MyDrive/ml_projects_new/parity_alpha_width_grid_sgd")
RUNS_DIR = DRIVE_ROOT / "runs"
PLOTS_DIR = DRIVE_ROOT / "plots"
ANALYSIS_DIR = DRIVE_ROOT / "analysis"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Runs: {RUNS_DIR}")
print(f"Plots: {PLOTS_DIR}")
print(f"Analysis: {ANALYSIS_DIR}")


Runs: /content/drive/MyDrive/ml_projects_new/parity_alpha_width_grid_sgd/runs
Plots: /content/drive/MyDrive/ml_projects_new/parity_alpha_width_grid_sgd/plots
Analysis: /content/drive/MyDrive/ml_projects_new/parity_alpha_width_grid_sgd/analysis


In [4]:
!rm -rf "$REPO_DIR"
!git clone "$GITHUB_REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!pip install -e .


Cloning into '/content/feature-learning-parity-task'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 129 (delta 75), reused 101 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 3.83 MiB | 44.52 MiB/s, done.
Resolving deltas: 100% (75/75), done.
/content/feature-learning-parity-task
Obtaining file:///content/feature-learning-parity-task
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for parity-net (pyproject.toml) ... done
  Created wheel for parity-net: filename=parity_net-0.1.0-0.editable-py3-none-any.whl size=3070 sha256=2dcbb7a69aacf6715edc4187137fb7278456dd312e5e181cfdea5360a7548ddf
  Stored in directory: /tmp/pip-ephem-wheel-cache-mpku16fu/wheels/a2/05/6b/5da726f55833c1e289bd27e

## Train Alpha-Width Grid

This cell writes one config per `(alpha, N)` pair and runs training. Existing final checkpoints are skipped unless `FORCE_RETRAIN = True`.

In [12]:
from fractions import Fraction
from pathlib import Path

import yaml

from parity_net.config import load_config
from parity_net.train import train

WIDTHS = [128, 256, 512, 1024, 2048]
ALPHAS = [Fraction(1, 1024) * (2**i) for i in range(10)]
FORCE_RETRAIN = False


def alpha_label(alpha: Fraction) -> str:
    return f"alpha_{alpha.numerator}_over_{alpha.denominator}"


def run_dir_for(alpha: Fraction, N: int) -> Path:
    return RUNS_DIR / alpha_label(alpha) / f"N_{N}"


def make_config(alpha: Fraction, N: int, output_dir: Path) -> dict:
    alpha_float = float(alpha)
    return {
        "model": {
            "input_dim": 32,
            "relevant_dim": 16,
            "N": N,
            "L": 3,
            "activation": "half-tanh",
            "use_readout_barrier": False,
            "embedding_weight_variance": 1.0 / 32,
            "freeze_embedding": False,
            "hidden_weight_variance": 1.0 / N,
            "readout_weight_variance": alpha_float / N,
            "use_post_activation_linear": False,
            "bias": False,
        },
        "training": {
            "num_steps": 150_000,
            "test_samples": 100_000,
            "batch_size": 512,
            "seed": 0,
            "device": "cuda",
            "dtype": "float32",
            "log_every": 1_000,
            "checkpoint_every": 1_000,
            "output_dir": str(output_dir),
            "barrier_c": None,
            "barrier_lambda": 10.0,
            "optimizer": {
                "name": "sgd",
                "lr": 3e-3,
                "lr_embedding": None,
                "lr_hidden": None,
                "lr_readout": None,
                "weight_decay": 1e-3,
                "wd_embedding": None,
                "wd_hidden": None,
                "wd_readout": None,
                "momentum": 0.9,
                "betas": [0.9, 0.999],
            },
        },
    }


grid_rows = []
config_paths = {}
for alpha in ALPHAS:
    for N in WIDTHS:
        run_dir = run_dir_for(alpha, N)
        run_dir.mkdir(parents=True, exist_ok=True)
        config = make_config(alpha, N, run_dir)
        config_path = run_dir / "config.yaml"
        with config_path.open("w") as f:
            yaml.safe_dump(config, f, sort_keys=False)
        config_paths[(alpha, N)] = config_path
        grid_rows.append(
            {
                "alpha": float(alpha),
                "alpha_fraction": str(alpha),
                "alpha_label": alpha_label(alpha),
                "N": N,
                "run_dir": str(run_dir),
                "config_path": str(config_path),
            }
        )

print(f"Prepared {len(grid_rows)} configs under {RUNS_DIR}")


Prepared 50 configs under /content/drive/MyDrive/ml_projects_new/parity_alpha_width_grid_sgd/runs


In [13]:
import pandas as pd

grid_df = pd.DataFrame(grid_rows)
display(grid_df)

,alpha,alpha_fraction,alpha_label,N,run_dir,config_path
0,0.000977,1/1024,alpha_1_over_1024,128,/content/drive/MyDrive/ml_projects_new/parity_...,/content/drive/MyDrive/ml_projects_new/parity_...
1,0.000977,1/1024,alpha_1_over_1024,256,/content/drive/MyDrive/ml_projects_new/parity_...,/content/drive/MyDrive/ml_projects_new/parity_...
2,0.000977,1/1024,alpha_1_over_1024,512,/content/drive/MyDrive/ml_projects_new/parity_...,/content/drive/MyDrive/ml_projects_new/parity_...
3,0.000977,1/1024,alpha_1_over_1024,1024,/content/drive/MyDrive/ml_projects_new/parity_...,/content/drive/MyDrive/ml_projects_new/parity_...
4,0.000977,1/1024,alpha_1_over_1024,2048,/content/drive/MyDrive/ml_projects_new/parity_...,/content/drive/MyDrive/ml_projects_new/parity_...
5,0.001953,1/512,alpha_1_over_512,128,/content/drive/MyDrive/ml_projects_new/parity_...,/content/drive/MyDrive/ml_projects_new/parity_...
6,0.001953,1/512,alpha_1_over_512,256,/content/drive/MyDrive/ml_projects_new/parity_...,/content/drive/MyDrive/ml_projects_new/parity_...
7,0.001953,1/512,alpha_1_over_512,512,/content/drive/MyDrive/ml_projects_new/parity_...,/content/drive/MyDrive/ml_projects_new/parity_...
8,0.001953,1/512,alpha_1_over_512,1024,/content/drive/MyDrive/ml_projects_new/parity_...,/content/drive/MyDrive/ml_projects_new/parity_...
9,0.001953,1/512,alpha_1_over_512,2048,/content/drive/MyDrive/ml_projects_new/parity_...,/content/drive/MyDrive/ml_projects_new/parity_...


In [ ]:
for alpha in ALPHAS:
    for N in WIDTHS:
        run_dir = run_dir_for(alpha, N)
        config_path = config_paths[(alpha, N)]
        final_checkpoint = run_dir / "checkpoints" / "final.pt"
        if final_checkpoint.exists() and not FORCE_RETRAIN:
            print(f"alpha={alpha}, N={N}: final checkpoint exists, skipping: {final_checkpoint}")
            continue

        print(f"alpha={alpha}, N={N}: training with {config_path}")
        train(load_config(config_path))


alpha=1/1024, N=128: training with /content/drive/MyDrive/ml_projects_new/parity_alpha_width_grid_sgd/runs/alpha_1_over_1024/N_128/config.yaml


training:   0%|          | 0/150000 [00:00<?, ?step/s]

{'step': 1000, 'elapsed_seconds': 3.9997738340000524, 'train_mse': 0.9677203893661499, 'barrier': 0.0, 'loss': 0.9677203893661499, 'test_mse': 0.9674330949783325, 'test_mse_d2': 0.9382858872413635, 'test_mse_d4': 1.0006790161132812, 'test_mse_d8': 1.0008128881454468, 'test_mse_d16': 1.000866413116455}
{'step': 2000, 'elapsed_seconds': 8.11016080200011, 'train_mse': 0.7192782163619995, 'barrier': 0.0, 'loss': 0.7192782163619995, 'test_mse': 0.7222895622253418, 'test_mse_d2': 0.47974225878715515, 'test_mse_d4': 0.9985710978507996, 'test_mse_d8': 1.0007599592208862, 'test_mse_d16': 1.0006014108657837}
{'step': 3000, 'elapsed_seconds': 12.221277762, 'train_mse': 0.5050095319747925, 'barrier': 0.0, 'loss': 0.5050095319747925, 'test_mse': 0.5067432522773743, 'test_mse_d2': 0.08297800272703171, 'test_mse_d4': 0.983521044254303, 'test_mse_d8': 1.0007715225219727, 'test_mse_d16': 1.001697301864624}
{'step': 4000, 'elapsed_seconds': 16.234197571000095, 'train_mse': 0.44523078203201294, 'barrier'

training:   0%|          | 0/150000 [00:00<?, ?step/s]

{'step': 1000, 'elapsed_seconds': 4.255481328999849, 'train_mse': 0.845971405506134, 'barrier': 0.0, 'loss': 0.845971405506134, 'test_mse': 0.8510180711746216, 'test_mse_d2': 0.7198480367660522, 'test_mse_d4': 1.0002782344818115, 'test_mse_d8': 1.001736044883728, 'test_mse_d16': 1.001901388168335}
{'step': 2000, 'elapsed_seconds': 8.587899034999737, 'train_mse': 0.4958155155181885, 'barrier': 0.0, 'loss': 0.4958155155181885, 'test_mse': 0.4949776530265808, 'test_mse_d2': 0.05842534825205803, 'test_mse_d4': 0.9882962107658386, 'test_mse_d8': 1.001524567604065, 'test_mse_d16': 1.0010271072387695}
{'step': 3000, 'elapsed_seconds': 12.839335111999844, 'train_mse': 0.4332660436630249, 'barrier': 0.0, 'loss': 0.4332660436630249, 'test_mse': 0.4337060749530792, 'test_mse_d2': 0.03297919034957886, 'test_mse_d4': 0.8091238737106323, 'test_mse_d8': 1.0019896030426025, 'test_mse_d16': 1.0012832880020142}
{'step': 4000, 'elapsed_seconds': 17.268363836999924, 'train_mse': 0.2692089080810547, 'barri

...

{'step': 1000, 'elapsed_seconds': 4.2168221560004895, 'train_mse': 0.47902941703796387, 'barrier': 0.0, 'loss': 0.47902941703796387, 'test_mse': 0.4807727634906769, 'test_mse_d2': 0.04072809964418411, 'test_mse_d4': 0.9642198085784912, 'test_mse_d8': 1.008268117904663, 'test_mse_d16': 1.0123505592346191}
{'step': 2000, 'elapsed_seconds': 8.531839353000578, 'train_mse': 0.25491538643836975, 'barrier': 0.0, 'loss': 0.25491538643836975, 'test_mse': 0.2534595727920532, 'test_mse_d2': 0.023226313292980194, 'test_mse_d4': 0.15067951381206512, 'test_mse_d8': 1.0052831172943115, 'test_mse_d16': 1.0027987957000732}
{'step': 3000, 'elapsed_seconds': 12.926864304999981, 'train_mse': 0.2147696316242218, 'barrier': 0.0, 'loss': 0.2147696316242218, 'test_mse': 0.21465657651424408, 'test_mse_d2': 0.014351249672472477, 'test_mse_d4': 0.022141963243484497, 'test_mse_d8': 1.0060288906097412, 'test_mse_d16': 1.0044130086898804}
{'step': 4000, 'elapsed_seconds': 17.329784862999986, 'train_mse': 0.20799003

training:   0%|          | 0/150000 [00:00<?, ?step/s]

{'step': 1000, 'elapsed_seconds': 8.605722638999396, 'train_mse': 0.35661306977272034, 'barrier': 0.0, 'loss': 0.35661306977272034, 'test_mse': 0.3594730794429779, 'test_mse_d2': 0.026579508557915688, 'test_mse_d4': 0.5274429321289062, 'test_mse_d8': 1.0167045593261719, 'test_mse_d16': 1.036279320716858}
{'step': 2000, 'elapsed_seconds': 17.73628725299932, 'train_mse': 0.21541985869407654, 'barrier': 0.0, 'loss': 0.21541985869407654, 'test_mse': 0.21699349582195282, 'test_mse_d2': 0.015247236005961895, 'test_mse_d4': 0.01897972635924816, 'test_mse_d8': 1.0264248847961426, 'test_mse_d16': 1.0041556358337402}
{'step': 3000, 'elapsed_seconds': 26.86511603300005, 'train_mse': 0.21115562319755554, 'barrier': 0.0, 'loss': 0.21115562319755554, 'test_mse': 0.2101515382528305, 'test_mse_d2': 0.010456552729010582, 'test_mse_d4': 0.010204046033322811, 'test_mse_d8': 1.008274793624878, 'test_mse_d16': 1.0112545490264893}
{'step': 4000, 'elapsed_seconds': 36.07696046100045, 'train_mse': 0.207081392

training:   0%|          | 0/150000 [00:00<?, ?step/s]

{'step': 1000, 'elapsed_seconds': 4.1166075000001, 'train_mse': 0.9671347737312317, 'barrier': 0.0, 'loss': 0.9671347737312317, 'test_mse': 0.9668102264404297, 'test_mse_d2': 0.9370205402374268, 'test_mse_d4': 1.0008021593093872, 'test_mse_d8': 1.0009199380874634, 'test_mse_d16': 1.0009407997131348}
{'step': 2000, 'elapsed_seconds': 8.615885170000183, 'train_mse': 0.7168967723846436, 'barrier': 0.0, 'loss': 0.7168967723846436, 'test_mse': 0.7200989723205566, 'test_mse_d2': 0.4755834639072418, 'test_mse_d4': 0.9986376166343689, 'test_mse_d8': 1.0008081197738647, 'test_mse_d16': 1.000649094581604}
{'step': 3000, 'elapsed_seconds': 13.241531966001276, 'train_mse': 0.5053529739379883, 'barrier': 0.0, 'loss': 0.5053529739379883, 'test_mse': 0.5071821212768555, 'test_mse_d2': 0.08264296501874924, 'test_mse_d4': 0.9858182668685913, 'test_mse_d8': 1.0007920265197754, 'test_mse_d16': 1.001731276512146}
{'step': 4000, 'elapsed_seconds': 17.402035888000682, 'train_mse': 0.450668603181839, 'barrie

## Heatmaps at a Given Step

Load saved `metrics.csv` files and plot heatmaps over the `(alpha, N)` grid for a chosen training step.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HEATMAP_STEP = "final"  # Use "final" or an integer logged step.
HEATMAP_METRICS = ["test_mse", "test_mse_d2", "test_mse_d4", "test_mse_d8", "test_mse_d16"]
HEATMAP_LOG_COLOR = False
HEATMAP_ANNOTATE = True


def select_metrics_row(metrics_df: pd.DataFrame, step):
    if metrics_df.empty:
        return None
    if step == "final":
        return metrics_df.iloc[-1]
    matches = metrics_df[metrics_df["step"] == step]
    if matches.empty:
        return None
    return matches.iloc[-1]


heatmap_rows = []
for alpha in ALPHAS:
    for N in WIDTHS:
        run_dir = run_dir_for(alpha, N)
        metrics_path = run_dir / "metrics.csv"
        if not metrics_path.exists():
            print(f"Missing metrics for alpha={alpha}, N={N}: {metrics_path}")
            continue
        metrics_df = pd.read_csv(metrics_path)
        row = select_metrics_row(metrics_df, HEATMAP_STEP)
        if row is None:
            print(f"No row for step={HEATMAP_STEP} in {metrics_path}")
            continue
        heatmap_rows.append(
            {
                "alpha": float(alpha),
                "alpha_fraction": str(alpha),
                "alpha_label": alpha_label(alpha),
                "N": N,
                "step": row["step"],
                **{metric: row.get(metric, np.nan) for metric in HEATMAP_METRICS},
            }
        )

heatmap_df = pd.DataFrame(heatmap_rows)
if heatmap_df.empty:
    print("No heatmap data found.")
else:
    display(heatmap_df.sort_values(["alpha", "N"]))
    step_label = "final" if HEATMAP_STEP == "final" else f"step_{int(HEATMAP_STEP):08d}"
    fig, axes = plt.subplots(1, len(HEATMAP_METRICS), figsize=(5 * len(HEATMAP_METRICS), 5), squeeze=False)
    alpha_order = list(reversed(ALPHAS))
    y_labels = [str(alpha) for alpha in alpha_order]
    for ax, metric in zip(axes[0], HEATMAP_METRICS):
        pivot = heatmap_df.pivot(index="alpha_fraction", columns="N", values=metric)
        values = np.full((len(alpha_order), len(WIDTHS)), np.nan)
        for row_idx, alpha in enumerate(alpha_order):
            for col_idx, N in enumerate(WIDTHS):
                alpha_key = str(alpha)
                if alpha_key in pivot.index and N in pivot.columns:
                    values[row_idx, col_idx] = pivot.loc[alpha_key, N]
        plot_values = values.copy()
        if HEATMAP_LOG_COLOR:
            plot_values = np.where(plot_values > 0, np.log10(plot_values), np.nan)
        im = ax.imshow(plot_values, aspect="auto", interpolation="nearest")
        ax.set_title(metric)
        ax.set_xticks(range(len(WIDTHS)))
        ax.set_xticklabels(WIDTHS, rotation=45)
        ax.set_yticks(range(len(alpha_order)))
        ax.set_yticklabels(y_labels)
        ax.set_xlabel("N")
        ax.set_ylabel("alpha")
        if HEATMAP_ANNOTATE:
            for row_idx in range(values.shape[0]):
                for col_idx in range(values.shape[1]):
                    value = values[row_idx, col_idx]
                    if np.isfinite(value):
                        ax.text(col_idx, row_idx, f"{value:.2g}", ha="center", va="center", color="white")
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("log10(MSE)" if HEATMAP_LOG_COLOR else "MSE")
    fig.suptitle(f"Metrics on alpha-width grid at {step_label}")
    fig.tight_layout()
    heatmap_path = PLOTS_DIR / f"alpha_width_heatmaps_{step_label}.png"
    fig.savefig(heatmap_path, dpi=150)
    print(f"Saved heatmaps to {heatmap_path}")
    plt.show()


## Curves for One Grid Point

Choose `(alpha, N)` and inspect the train/test curves from that saved run.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

CURVE_ALPHA = Fraction(1, 64)
CURVE_N = 512
CURVE_CHECKPOINT_STEP = "final"  # Used to print the matching checkpoint path.
CURVE_LOG_X = False
CURVE_LOG_Y = False


def checkpoint_name(checkpoint_step):
    if checkpoint_step == "final":
        return "final.pt", "final"
    if isinstance(checkpoint_step, int):
        return f"step_{checkpoint_step:08d}.pt", f"step_{checkpoint_step:08d}"
    raise ValueError('checkpoint_step must be "final" or an integer')


curve_run_dir = run_dir_for(CURVE_ALPHA, CURVE_N)
curve_metrics_path = curve_run_dir / "metrics.csv"
curve_checkpoint_file, curve_checkpoint_label = checkpoint_name(CURVE_CHECKPOINT_STEP)
curve_checkpoint_path = curve_run_dir / "checkpoints" / curve_checkpoint_file
print(f"Run directory: {curve_run_dir}")
print(f"Checkpoint path: {curve_checkpoint_path}")

if not curve_metrics_path.exists():
    raise FileNotFoundError(f"Missing metrics: {curve_metrics_path}")
curve_df = pd.read_csv(curve_metrics_path)
display(curve_df.tail())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
if "train_mse" in curve_df:
    ax.plot(curve_df["step"], curve_df["train_mse"], label="train_mse")
ax.plot(curve_df["step"], curve_df["test_mse"], label="test_mse")
ax.set_xlabel("Step")
ax.set_ylabel("MSE")
ax.set_title(f"Train/Test MSE, alpha={CURVE_ALPHA}, N={CURVE_N}")
if CURVE_LOG_X:
    positive_steps = curve_df["step"] > 0
    if positive_steps.any():
        ax.set_xscale("log")
if CURVE_LOG_Y:
    ax.set_yscale("log")
ax.grid(True, alpha=0.3, which="both")
ax.legend()

ax = axes[1]
for metric in ["test_mse", "test_mse_d2", "test_mse_d4", "test_mse_d8", "test_mse_d16"]:
    if metric in curve_df:
        ax.plot(curve_df["step"], curve_df[metric], label=metric)
ax.set_xlabel("Step")
ax.set_ylabel("MSE")
ax.set_title("Test MSE by degree")
if CURVE_LOG_X:
    positive_steps = curve_df["step"] > 0
    if positive_steps.any():
        ax.set_xscale("log")
if CURVE_LOG_Y:
    ax.set_yscale("log")
ax.grid(True, alpha=0.3, which="both")
ax.legend()
fig.tight_layout()
curve_plot_path = PLOTS_DIR / f"curves_alpha_{alpha_label(CURVE_ALPHA)}_N_{CURVE_N}_{curve_checkpoint_label}.png"
fig.savefig(curve_plot_path, dpi=150)
print(f"Saved curves to {curve_plot_path}")
plt.show()


## Final-Checkpoint PCA and Weight-Variance Analysis

For a chosen `(alpha, N)` and checkpoint, compute effective residual-stream dimensions, PCA intervention MSE curves, and trained weight variances.

In [ ]:
import torch

from parity_net.analysis import (
    collect_layer_activations,
    make_pca_intervention,
    pca_from_activations,
    per_degree_mse,
    predict_in_batches,
    rank_for_threshold,
)
from parity_net.checkpoint import load_checkpoint
from parity_net.data import ParityDataset, load_dataset
from parity_net.train import resolve_device, resolve_dtype

PCA_ALPHA = Fraction(1, 64)
PCA_N = 512
PCA_CHECKPOINT_STEP = "final"  # Use "final" or an integer step.
PCA_SAMPLES = 20_000
ANALYSIS_LAYER_IDX = 2  # 0 is embedding, later indices are after residual blocks.
KEEP_PCS_MIN = 0
KEEP_PCS_MAX = 128
KEEP_PCS_STEP = 1
USE_LOG_PCA_MSE_AXIS = False


def assert_finite_tensor(tensor, label):
    if not torch.isfinite(tensor).all().item():
        raise ValueError(f"{label} contains NaN or Inf")


pca_checkpoint_file, pca_checkpoint_label = checkpoint_name(PCA_CHECKPOINT_STEP)
pca_run_dir = run_dir_for(PCA_ALPHA, PCA_N)
pca_checkpoint_path = pca_run_dir / "checkpoints" / pca_checkpoint_file
if not pca_checkpoint_path.exists():
    raise FileNotFoundError(f"Missing checkpoint: {pca_checkpoint_path}")

load_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, payload, _ = load_checkpoint(pca_checkpoint_path, load_device)
training = payload["config"]["training"]
device = resolve_device(training["device"])
dtype = resolve_dtype(training["dtype"])
model = model.to(device=device, dtype=dtype)
model.eval()

test_data_path = Path(payload.get("test_data_path") or pca_run_dir / "test_data.pt")
if not test_data_path.exists():
    raise FileNotFoundError(f"Missing saved test data: {test_data_path}")
test_data = load_dataset(test_data_path, device, dtype)
if PCA_SAMPLES < test_data.x.shape[0]:
    heldout = ParityDataset(x=test_data.x[:PCA_SAMPLES], y=test_data.y[:PCA_SAMPLES])
else:
    heldout = test_data
batch_size = int(training["batch_size"])

analysis_output_dir = ANALYSIS_DIR / alpha_label(PCA_ALPHA) / f"N_{PCA_N}" / pca_checkpoint_label
analysis_output_dir.mkdir(parents=True, exist_ok=True)

pred = predict_in_batches(model, heldout.x, batch_size)
assert_finite_tensor(pred, "baseline predictions")
baseline_metrics = per_degree_mse(pred, heldout.y)
baseline_df = pd.DataFrame([{ "alpha": float(PCA_ALPHA), "alpha_fraction": str(PCA_ALPHA), "N": PCA_N, **baseline_metrics }])
baseline_df.to_csv(analysis_output_dir / "baseline_mse.csv", index=False)

activations = collect_layer_activations(model, heldout.x, batch_size)
for layer_idx, activations_for_layer in enumerate(activations):
    assert_finite_tensor(activations_for_layer, f"layer {layer_idx} activations")

pcas = [pca_from_activations(layer_acts) for layer_acts in activations]
rank_rows = []
for layer_idx, pca in enumerate(pcas):
    cumulative = pca["cumulative_explained_variance"]
    rank_rows.append(
        {
            "alpha": float(PCA_ALPHA),
            "alpha_fraction": str(PCA_ALPHA),
            "N": PCA_N,
            "checkpoint": pca_checkpoint_label,
            "layer_idx": layer_idx,
            "rank_90": rank_for_threshold(cumulative, 0.90),
            "rank_99": rank_for_threshold(cumulative, 0.99),
            "num_dimensions": cumulative.numel(),
        }
    )
rank_df = pd.DataFrame(rank_rows)
rank_df.to_csv(analysis_output_dir / "pca_rank_thresholds.csv", index=False)

if ANALYSIS_LAYER_IDX < 0 or ANALYSIS_LAYER_IDX >= len(pcas):
    raise ValueError(f"ANALYSIS_LAYER_IDX must be in [0, {len(pcas) - 1}]")
max_keep_pcs = min(KEEP_PCS_MAX, pcas[ANALYSIS_LAYER_IDX]["components"].shape[0])
keep_values = list(range(KEEP_PCS_MIN, max_keep_pcs + 1, KEEP_PCS_STEP))
if keep_values[-1] != max_keep_pcs:
    keep_values.append(max_keep_pcs)

pca_sweep_rows = []
for keep_pcs in keep_values:
    intervention = make_pca_intervention(pcas[ANALYSIS_LAYER_IDX], keep_pcs)
    pred_intervened = predict_in_batches(
        model,
        heldout.x,
        batch_size,
        intervention=(ANALYSIS_LAYER_IDX, intervention),
    )
    assert_finite_tensor(pred_intervened, f"intervened predictions keep_pcs={keep_pcs}")
    pca_sweep_rows.append(
        {
            "alpha": float(PCA_ALPHA),
            "alpha_fraction": str(PCA_ALPHA),
            "N": PCA_N,
            "checkpoint": pca_checkpoint_label,
            "layer_idx": ANALYSIS_LAYER_IDX,
            "keep_pcs": keep_pcs,
            **per_degree_mse(pred_intervened, heldout.y),
        }
    )
pca_sweep_df = pd.DataFrame(pca_sweep_rows)
pca_sweep_df.to_csv(analysis_output_dir / "pca_intervention_mse.csv", index=False)

weight_variance_df = pd.DataFrame(
    [
        {
            "alpha": float(PCA_ALPHA),
            "alpha_fraction": str(PCA_ALPHA),
            "N": PCA_N,
            "checkpoint": pca_checkpoint_label,
            "layer": layer,
            "variance": variance,
        }
        for layer, variance in model.weight_variances().items()
    ]
)
weight_variance_df.to_csv(analysis_output_dir / "weight_variances.csv", index=False)

print(f"Saved analysis to {analysis_output_dir}")
display(baseline_df)
display(rank_df)
display(pca_sweep_df.head())
display(weight_variance_df)


## Summary Plots

Plot PCA effective dimensions by layer, PCA intervention curves, and trained weight variances for the selected grid point.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(rank_df["layer_idx"], rank_df["rank_90"], marker="o", label="90% variance")
ax.plot(rank_df["layer_idx"], rank_df["rank_99"], marker="o", label="99% variance")
ax.set_xlabel("Layer index")
ax.set_ylabel("Number of PCs")
ax.set_title(f"Effective dimension, alpha={PCA_ALPHA}, N={PCA_N}")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
effective_dim_path = PLOTS_DIR / f"effective_dimension_alpha_{alpha_label(PCA_ALPHA)}_N_{PCA_N}_{pca_checkpoint_label}.png"
fig.savefig(effective_dim_path, dpi=150)
print(f"Saved {effective_dim_path}")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for metric in ["mse_all", "mse_d2", "mse_d4", "mse_d8", "mse_d16"]:
    if metric in pca_sweep_df:
        ax.plot(pca_sweep_df["keep_pcs"], pca_sweep_df[metric], marker="o", label=metric)
ax.set_xlabel("Number of PCs kept")
ax.set_ylabel("MSE")
ax.set_title(f"PCA intervention, layer {ANALYSIS_LAYER_IDX}, alpha={PCA_ALPHA}, N={PCA_N}")
if USE_LOG_PCA_MSE_AXIS:
    ax.set_yscale("log")
ax.grid(True, alpha=0.3, which="both")
ax.legend()
fig.tight_layout()
pca_plot_path = PLOTS_DIR / f"pca_intervention_alpha_{alpha_label(PCA_ALPHA)}_N_{PCA_N}_{pca_checkpoint_label}_layer_{ANALYSIS_LAYER_IDX}.png"
fig.savefig(pca_plot_path, dpi=150)
print(f"Saved {pca_plot_path}")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(weight_variance_df["layer"], weight_variance_df["variance"])
ax.set_yscale("log")
ax.set_xlabel("Weight matrix")
ax.set_ylabel("Trained weight variance")
ax.set_title(f"Weight variances, alpha={PCA_ALPHA}, N={PCA_N}")
plt.xticks(rotation=45, ha="right")
ax.grid(True, alpha=0.3, axis="y", which="both")
fig.tight_layout()
weight_plot_path = PLOTS_DIR / f"weight_variances_alpha_{alpha_label(PCA_ALPHA)}_N_{PCA_N}_{pca_checkpoint_label}.png"
fig.savefig(weight_plot_path, dpi=150)
print(f"Saved {weight_plot_path}")
plt.show()
